# Priority Queue Contracts and Heap Shape


```{contents}
:local:
:depth: 2
```


```{index} Priority Describes Removal, Not a Fully Sorted Array
```



```{index} Priority Describes Removal, Not a Fully Sorted Array
```

## Priority Describes Removal, Not a Fully Sorted Array



A service desk chooses the lowest numbered priority next. That is a removal contract. It does not require that every stored element be maintained in complete sorted order, nor does it say that equal priorities must leave in arrival order. State the priority direction and tie policy before interpreting a printed storage array.

A min-heap requires each parent's priority to be no greater than its children's. The root is therefore a minimum, but siblings need not be ordered. In the array [1, 4, 2, 7, 6, 3], the values 4 and 2 can appear in that order without violating the heap. Reading the array from left to right is not the same as repeatedly removing the minimum.

| Operation needed | Heap property helps with it? | Reason |
| --- | --- | --- |
| Inspect the minimum | Yes | It is at the root |
| Remove the minimum and restore the property | Yes | Only a repair path is needed |
| Print everything in sorted order without changing storage | Not directly | Heap order is weaker than total order |
| Find an arbitrary ID | Not directly | Priority ancestry does not locate arbitrary keys |

The complete-tree shape lets an array encode parent/child positions without one node object per edge. With zero-based positions, a non-root index i has parent (i-1)/2 using integer division; its potential children are 2i+1 and 2i+2. A child position must still be checked against Count. Capacity is not the number of live heap entries.

Guided drawing: place the six array values above into the implicit tree. Check each parent-child relation. Then compare positions one and two: their values are out of sorted-array order but are allowed as siblings. This single counterexample explains why ordinary binary search cannot be applied to heap storage.

For related explanation, see the further reading below.[^development-reading]


```{index} Priority Queue Behavior
```

## Priority Queue Behavior




```{index} priority queue
```


A **priority queue** removes the item with the next priority, not necessarily the item that arrived first. In C#'s `PriorityQueue<TElement, TPriority>`, lower priority values come out first by default.


In [1]:
using System;
using System.Collections.Generic;

var tasks = new PriorityQueue<string, int>();
tasks.Enqueue("Process invoice", 2);
tasks.Enqueue("Respond to outage", 0);
tasks.Enqueue("Send summary", 3);
tasks.Enqueue("Reset password", 1);

while (tasks.Count > 0)
{
    string task = tasks.Dequeue();
    Console.WriteLine(task);
}


Respond to outage
Reset password
Process invoice
Send summary


The outage has priority `0`, so it is processed before lower-urgency work. Priority queues are useful for schedulers, event simulations, pathfinding, triage, and any workflow where "next" means most important rather than oldest.


**Common mistake.** Expecting `foreach (var item in tasks)` to list the work in priority order. Enumeration follows the internal heap layout, which is not sorted; only repeated `Dequeue` calls deliver items by priority.


A priority queue defines how you select the next element. It does not promise that enumeration is sorted or that arbitrary keys are easy to locate. The numerical priority has an application meaning: here zero is most urgent. Dequeue from an empty queue is an error; `TryDequeue` reports false instead.

Equal priorities need a policy. The .NET library does not guarantee arrival order for ties. If the service desk requires first-arrived service among equally urgent tickets, use the pair `(urgency, arrivalNumber)` as the priority. Tuple comparison considers urgency first and arrival second. Each arrival number is unique within the queue, so the resulting order is deterministic.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var queue = new PriorityQueue<string, (int Urgency, long Arrival)>();
long arrival = 0;
queue.Enqueue("T-100", (2, arrival++));
queue.Enqueue("T-101", (0, arrival++));
queue.Enqueue("T-102", (2, arrival++));
while (queue.TryDequeue(out string? ticket, out var priority))
    Console.WriteLine($"{ticket}: urgency={priority.Urgency}, arrival={priority.Arrival}");
Console.WriteLine($"empty={queue.Count == 0}");

T-101: urgency=0, arrival=1
T-100: urgency=2, arrival=0
T-102: urgency=2, arrival=2
empty=True


The order is T-101, T-100, T-102. A later urgent ticket can move ahead, while equal-urgency tickets retain their arrival order. The queue alone does not prevent starvation if urgent work arrives indefinitely. Fairness requires an additional scheduling policy.

The manual implementation in this section is a **binary** heap. The current .NET `PriorityQueue` uses four children per parent internally. Its API behavior is the same priority-queue idea, but the binary index formulas below describe the teaching implementation, not the library's internal layout. [Microsoft's PriorityQueue documentation](https://learn.microsoft.com/en-us/dotnet/api/system.collections.generic.priorityqueue-2) describes both tie behavior and the library representation.

```{index} Heap Representation
```

## Heap Representation




```{index} heap; min-heap; complete tree
```


A **heap** is a common priority-queue representation. A binary heap has two key rules:

- **Shape rule**: the tree is complete, filled level by level from left to right.
- **Order rule**: in a min-heap, each parent priority is less than or equal to its child priorities.

Because the shape is complete, a heap can be stored compactly in an array. For an item at index `i`, the children are at `2 * i + 1` and `2 * i + 2`.


In [1]:
using System;

int[] heap = { 1, 3, 2, 7, 5, 6, 4 };

for (int parent = 0; parent < heap.Length; parent++)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;

    if (left < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> left child {heap[left]}");
    }

    if (right < heap.Length)
    {
        Console.WriteLine($"{heap[parent]} -> right child {heap[right]}");
    }
}


1 -> left child 3
1 -> right child 2
3 -> left child 7
3 -> right child 5
2 -> left child 6
2 -> right child 4


The array is not fully sorted. It only guarantees that each parent is no larger than its children. That weaker rule is enough to keep the minimum at the root.


**Common mistake.** Checking the heap rule only at the root, or comparing siblings. The order rule relates each parent to its own children: in `1, 3, 2, 7, 5, 6, 4` the siblings 3 and 2 are out of numeric order and the heap is still valid, while a single parent larger than one of its children would invalidate it.


For a non-root index i, its parent is `(i - 1) / 2`, using integer division. Valid child indexes must be less than Count. Count describes occupied positions; the backing array may have extra capacity that is not part of the heap.

```text
          1 (0)
        /       \
     3 (1)     2 (2)
    /    \     /    \
 7 (3) 5 (4) 6 (5) 4 (6)
```

The values obey parent-child order, but siblings and separate subtrees can be out of numerical order. For example, 3 precedes 2 in the array. Repeatedly removing the root produces a sorted sequence; reading the raw array does not. The complete shape guarantees height Θ(log n) for a nonempty binary heap.

A priority queue is an access rule. The next section implements it with a binary heap, while application code may use the library PriorityQueue. Enumeration is not a sorted report; repeated removal is the operation whose priority order is specified.


```{index} Practice: Service the Minimum Priority
```

## Practice: Service the Minimum Priority




```{index} Practice: Service the Minimum Priority
```


Complete the operation, run the cell, and explain why the chosen item is returned.


In [ ]:
using System;
using System.Collections.Generic;

PriorityQueue<string, int> queue = new PriorityQueue<string, int>();
queue.Enqueue("Routine", 5); queue.Enqueue("Urgent", 1);
// TODO: explain the result.
Console.WriteLine(queue.Dequeue());


In [1]:
using System;
using System.Collections.Generic;

PriorityQueue<string, int> queue = new PriorityQueue<string, int>();
queue.Enqueue("Routine", 5); queue.Enqueue("Urgent", 1);
Console.WriteLine(queue.Dequeue());


Urgent


The output is `Urgent`. Its priority 1 is lower than Routine's 5, so it leaves first even though Routine was enqueued first; arrival order does not matter here. Swap the two priorities and `Routine` comes out instead.


```{rubric} Footnotes
```


[^1801_priority_queues-1]: Equal library priorities do not promise first-in-first-out ties. Supply a secondary arrival number when that is a requirement.


[^development-reading]: Further reading: [OpenDSA CS3, Heaps and Priority Queues](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Heaps.html). The worked business scenarios and guided reasoning here are original teaching examples.
